## Goal: get dsa <-> PF muon cross references working

The goal is to edit the schema to be able to access the DSA muons matched to PF muons (and vice versa). 

Running on Coffea2025.5.rc build (release candidate).

Accomplished:
1. Using NanoEventsFactory, test the custom NanoAOD schema to add the cross references between DSA and PF muons and add a "matched_muons" collection to access the muons that correspond to DSAMuon_muonMatchNidx branches

Remaining steps:
1. Create a stripped-down version of the schema, that only adds what we need on top of the standard NanoAODSchema
2. Update the processor so that the fields are handled correctly when building up the lepton jets
3. Test that the selection works as expected when applying masks. Make sure the global indices do not reach across events after applying selections to the target collection (to get this to work for real)
4. Figure out how to attach the info in the muonMatchN branches to correspond to the muons in matched_muons (so we can check the number of matched segments for muons passing our selections)
5. Figure out how to change the type of the LLPNanoAOD idx branches from float to int32 (to avoid having to modify coffea). See note.

*NOTE:* Right now, the code only works if you comment out [these lines](https://github.com/scikit-hep/coffea/blob/539c4b58961b7bae86038caf67f1c7db83dded3a/coffea/nanoevents/transforms.py#L129-L130) in your local coffea install. It would be better to change the type of the muonMatchNidx branches to int32 (it's currently a float for some reason), but I haven't figured out how. 


# Test with NanoEventsFactory

In [1]:
# python
# python
import sys
import os
import importlib
# columnar analysis
from coffea.nanoevents import NanoEventsFactory, NanoAODSchema
from coffea import processor
# local
sidm_path = str(os.getcwd()).split("/sidm")[0]
if sidm_path not in sys.path: sys.path.insert(1, sidm_path)
from sidm.tools import llpnanoaodschema, utilities
from sidm.tools import full_nano
# always reload local modules to pick up changes during development
importlib.reload(llpnanoaodschema)
importlib.reload(utilities)

/tmp/ipykernel_2429/4181269814.py:7: DeprecationWarning: NanoEventsFactory.from_root() behavior has changed.
    The default behavior is that now it reads the input root file using
    the newly developed virtual arrays backend of awkward instead of dask.
    The backend choice is controlled by the `mode` argument of the method
    which can be set to "eager", "virtual", or "dask".
    The new default is "virtual" while the `delayed` argument has been removed.
    The old `delayed=True` is now equivalent to `mode="dask"`.
    The old `delayed=False` is now equivalent to `mode="eager"`.
    
  from coffea.nanoevents import NanoEventsFactory, NanoAODSchema


<module 'sidm.tools.utilities' from '/home/cms-jovyan/SIDM_forks_2025/dsaMatching/SIDM/sidm/tools/utilities.py'>

In [2]:
samples = [
    '2Mu2E_500GeV_5p0GeV_8p0mm',
]
fileset = utilities.make_fileset(samples, "llpNanoAOD_v2", max_files=1, location_cfg="signal_2mu2e_v10.yaml")
# create events collection from single file
fname = fileset[samples[0]]["files"][0]

In [3]:
import uproot
treepath="Events"
branch_names = ["DSAMuon_muonMatch1idx","Jet_muonIdx1"]
with uproot.open(f"{fname}:{treepath}") as tree:
    for branch_name in branch_names:
        branch = tree[branch_name]
        # The 'form' describes the structure and primitive type
        print(f"\n--- Inspecting branch: '{branch_name}' in '{fname}:{treepath}' ---")
        print(f"Branch interpretation (form): {branch.interpretation}")




--- Inspecting branch: 'DSAMuon_muonMatch1idx' in 'root://xcache//store/group/lpcmetx/SIDM/ULSignalSamples/2018_v10/BsTo2DpTo2Mu2e/CutDecayFalse_SIDM_BsTo2DpTo2Mu2e_MBs-500_MDp-5p0_ctau-8p0_v3/LLPnanoAODv2/CutDecayFalse_SIDM_BsTo2DpTo2Mu2e_MBs-500_MDp-5p0_ctau-8p0_v3_part-0.root:Events' ---
Branch interpretation (form): AsJagged(AsDtype('>f4'))

--- Inspecting branch: 'Jet_muonIdx1' in 'root://xcache//store/group/lpcmetx/SIDM/ULSignalSamples/2018_v10/BsTo2DpTo2Mu2e/CutDecayFalse_SIDM_BsTo2DpTo2Mu2e_MBs-500_MDp-5p0_ctau-8p0_v3/LLPnanoAODv2/CutDecayFalse_SIDM_BsTo2DpTo2Mu2e_MBs-500_MDp-5p0_ctau-8p0_v3_part-0.root:Events' ---
Branch interpretation (form): AsJagged(AsDtype('>i4'))


In [4]:
factory_llp = NanoEventsFactory.from_root(
    {fname: "Events"},
    schemaclass=full_nano.NanoAODSchema
    #schemaclass=NanoAODSchema
    #schemaclass=llpnanoaodschema.LLPNanoAODSchema,
)

/home/cms-jovyan/SIDM_forks_2025/dsaMatching/SIDM/sidm/tools/full_nano.py:350: RuntimeWarning: Missing cross-reference index for LowPtElectron_electronIdx => Electron
  warnings.warn(
/home/cms-jovyan/SIDM_forks_2025/dsaMatching/SIDM/sidm/tools/full_nano.py:350: RuntimeWarning: Missing cross-reference index for LowPtElectron_photonIdx => Photon
  warnings.warn(


In [5]:
type(factory_llp)

coffea.nanoevents.factory.NanoEventsFactory

In [6]:
events_llp = factory_llp.events()

In [7]:
pf = events_llp.Muon
dsa = events_llp.DSAMuon
jet = events_llp.Jet
#Check if matched_muons is included in the DSAMuon collection
all_attrs = dir(dsa)
print([attr for attr in all_attrs if attr.startswith('mat')])

#print(dsa)
#dsa.fields

['matched_muons']


The following command (specifically materializing the dsa.muonIdxG array) will only work if you comment out [these lines](https://github.com/scikit-hep/coffea/blob/539c4b58961b7bae86038caf67f1c7db83dded3a/coffea/nanoevents/transforms.py#L129-L130) in your local coffea install (you might have to add it on each coffea casa node). It would be better to change the type of the `muonMatchNidx` branches to `int32` (it's currently a float for some reason), but I haven't figured out how.

In [8]:
import awkward as ak
ak.materialize(dsa.muonMatch1)
ak.materialize(dsa.muonMatch1idx)
ak.materialize(dsa.pt)
ak.materialize(pf.pt)
ak.materialize(dsa.muonMatch4idx)

ak.materialize(dsa.muonIdxG)
#ak.materialize(dsa.muonMatch1idxG)

<Array [[[1, 0, -1, -1, -1], [...]], ...] type='4364 * var * var * int64[pa...'>

In [9]:
print(dsa.muonMatch1idx[0,0].dtype)

float32


In event 1, we have three dsa muons and 2 pf muons. The first dsa muon matches the 2nd pf muon and vice versa.

In [10]:
print("DSA pt in events 0-2:", dsa.pt[0:3].to_list())
print("PF pt in events 0-2:", pf[0:3].pt.to_list())
#print(dsa.muonMatch1idxG[1]) #don't want to use this directly; numbers aren't easily interpretable
print("ID of PF muons matched to DSA muons in events 0-2:",dsa.muonIdxG[0:3].to_list()) #global index (for the first 5 matches)
print("pt of PF muons matched to DSA muons in event 1:",dsa.matched_muons.pt[1].to_list())

DSA pt in events 0-2: [[90.20301055908203, 169.33285522460938], [5.917826175689697, 321.9776916503906, 1.6178698539733887], [161.76913452148438, 24.506929397583008]]
PF pt in events 0-2: [[717.1842651367188, 135.23300170898438], [212.15811157226562, 5.346219539642334], [197.0467071533203, 21.183835983276367]]
ID of PF muons matched to DSA muons in events 0-2: [[[1, 0, -1, -1, -1], [0, 1, -1, -1, -1]], [[3, 2, -1, -1, -1], [2, 3, -1, -1, -1], [3, 2, -1, -1, -1]], [[4, 5, -1, -1, -1], [5, 4, -1, -1, -1]]]
pt of PF muons matched to DSA muons in event 1: [[5.346219539642334, 212.15811157226562, None, None, None], [212.15811157226562, 5.346219539642334, None, None, None], [5.346219539642334, 212.15811157226562, None, None, None]]


Great, it appears matched_muons allows me to access the muons that correspond to `muonMatchNidx` branches. Next I'll check what happens after applying a selection to the muons

# New test: run in processor

Status: not currently working

```
File ~/SIDM_forks_2025/dsaMatching/SIDM/sidm/tools/sidm_processor.py:191, in SidmProcessor.build_lepton_jets(self, objs, lj_reco)
    190 photon_inputs = self.make_vector(objs, "photons", all_fields, type_id=4)
--> 191 lj_inputs = ak.concatenate([muon_inputs, dsa_inputs, ele_inputs, photon_inputs], axis=-1)
    193 distance_param = abs(lj_reco)

ValueError: arrays to concatenate do not have the same depth for negative axis=-1
```

In [11]:
# python
import sys, os
import importlib
# columnar analysis
from coffea.nanoevents import NanoAODSchema
from coffea import processor
# local
sidm_path = str(os.getcwd()).split("/sidm")[0]
if sidm_path not in sys.path: sys.path.insert(1, sidm_path)
from sidm.tools import sidm_processor, utilities
from sidm.tools import full_nano
#from sidm.tools import llpnanoaodschema_0726 as llpnanoaodschema
# always reload local modules to pick up changes during development
importlib.reload(sidm_processor)
importlib.reload(utilities)
# plotting
import matplotlib.pyplot as plt
utilities.set_plot_style()

In [12]:
samples = [
    '2Mu2E_500GeV_5p0GeV_8p0mm',
]
fileset = utilities.make_fileset(samples, "llpNanoAOD_v2", max_files=1, location_cfg="signal_2mu2e_v10.yaml")

In [13]:
runner = processor.Runner(
    executor=processor.IterativeExecutor(),
    schema=full_nano.NanoAODSchema,
    #schema=llpnanoaodschema.NanoAODSchema,
    maxchunks=1,
    skipbadfiles=True,
)

channels = [
    "2mu2e",
]
p = sidm_processor.SidmProcessor(
    channels,
    ["base"],
    verbose=True,
)

output = runner.run(fileset, treename='Events', processor_instance=p)
out = output["out"]

Output()

Output()

/home/cms-jovyan/SIDM_forks_2025/dsaMatching/SIDM/sidm/tools/full_nano.py:350: RuntimeWarning: Missing 
cross-reference index for LowPtElectron_electronIdx => Electron
  warnings.warn(

/home/cms-jovyan/SIDM_forks_2025/dsaMatching/SIDM/sidm/tools/full_nano.py:350: RuntimeWarning: Missing 
cross-reference index for LowPtElectron_photonIdx => Photon
  warnings.warn(

/usr/local/lib/python3.12/site-packages/awkward/_nplikes/array_module.py:292: RuntimeWarning: invalid value 
encountered in divide
  return impl(*broadcasted_args, **(kwargs or {}))

...

Applying dsaMuons |eta| < 2.4

Applying dsaMuons displaced ID

Applying dsaMuons dR(dsa, pf) > 0.2

Warning: Unable to apply dR(dsa, pf) > 0.2 for dsaMuons. Skipping.

Exception: Failed processing file: WorkItem(dataset='2Mu2E_500GeV_5p0GeV_8p0mm', filename='root://xcache//store/group/lpcmetx/SIDM/ULSignalSamples/2018_v10/BsTo2DpTo2Mu2e/CutDecayFalse_SIDM_BsTo2DpTo2Mu2e_MBs-500_MDp-5p0_ctau-8p0_v3/LLPnanoAODv2/CutDecayFalse_SIDM_BsTo2DpTo2Mu2e_MBs-500_MDp-5p0_ctau-8p0_v3_part-0.root', treename='Events', entrystart=0, entrystop=4364, fileuuid=b'\xac\xd8R\xc6\xbf\x88\x11\xef\x9b\xf4[\x15\xe6\x9b\xbe\xef', usermeta={'skim_factor': 1.0})

# Junk

In [ ]:
gen = events_llp.GenPart
gen.matched_genpart

In [ ]:
jet.fields

In [ ]:
jet.matched_muons?


In [ ]:
ak.materialize(jet.muonIdx1G)

In [ ]:
ak.materialize(jet.matched_muons.pt)

In [ ]:
electron= events_llp.Electron
ak.materialize(electron.matched_jet.pt)

In [ ]:
electron.fields